# Corpus description

The filtered corpus, on its sung voices: the numbers of the report's section on length, rhythm and texture. Reads `data/interim/00_pieces.csv` and `01_corpus.csv`; writes `tex/figures/03_texture.pdf`.

In [ ]:
import os
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
import numpy as np
import pandas as pd
from music21 import converter
from figures import stacked
%config InlineBackend.print_figure_kwargs = {"bbox_inches": None}

In [ ]:
RAW = Path("../data/raw")
INTERIM = Path("../data/interim")
WORKERS = os.cpu_count() // 2
BEAT = {"bach": 1, "palestrina": 2}   # the beat in quarter notes: a quarter in 4/4, a half note in 4/2

corpus = pd.read_csv(INTERIM / "01_corpus.csv").set_index("file")
pieces = pd.read_csv(INTERIM / "00_pieces.csv").set_index("file")
corpus = corpus.join(pieces[["measures", "duration"]])   # the filters do not change a file's length; its notes come from the voices
assert corpus[["measures", "duration"]].notna().all().all()
corpus.groupby("corpus").size().rename("files").to_frame()

## Length, voices and rhythm

- Bars (four beats in both), voices, and notes per voice per beat (a tied note counts once); median and quartiles.

In [ ]:
def attacks_of(path, parts):   # notes that the voices start: a tied note counts once
    score = converter.parse(path)
    return sum(n.tie is None or n.tie.type == "start" for p in parts for n in score.parts[p].flatten().notes)

paths = [RAW / c / f for f, c in zip(corpus.index, corpus["corpus"])]
parts = [[int(p) for p in s.split("|")] for s in corpus["voice_parts"]]
with ProcessPoolExecutor(max_workers=WORKERS) as pool:
    corpus["attacks"] = list(pool.map(attacks_of, paths, parts, chunksize=8))
assert (corpus["attacks"] > 0).all() and (corpus["attacks"] <= corpus["voice_notes"]).all()
corpus["rate"] = corpus["attacks"] / corpus["voices"] / (corpus["duration"] / corpus["corpus"].map(BEAT))   # notes per voice per beat
corpus.groupby("corpus")[["measures", "voices", "rate"]].quantile([0.25, 0.5, 0.75]).T.round(2)

**Result:** medians, Palestrina against Bach: 41 bars against 14, 5 voices against 4, 0.59 notes per voice per beat against 1.10.

## Texture

- **Measure:** how many sung voices sound in each grid cell (a quarter of the beat): every voice, all but one, or fewer.
- **Left out:** moments when every voice rests, counted apart.

In [ ]:
def sounding_of(path, parts, step):   # grid steps with no voice, fewer than all but one, all but one, and every voice sounding
    score = converter.parse(path)
    grid = np.arange(0, float(score.highestTime), step)
    sounding = np.zeros((len(parts), len(grid)), dtype=bool)
    for v, p in enumerate(parts):
        for n in score.parts[p].flatten().notes:
            sounding[v, (grid >= float(n.offset)) & (grid < float(n.offset + n.quarterLength))] = True
    singing, n = sounding.sum(axis=0), len(parts)
    return [(singing == 0).sum(), ((singing > 0) & (singing < n - 1)).sum(), (singing == n - 1).sum(), (singing == n).sum()]

steps = [BEAT[c] / 4 for c in corpus["corpus"]]
with ProcessPoolExecutor(max_workers=WORKERS) as pool:
    counts = list(pool.map(sounding_of, paths, parts, steps, chunksize=8))
totals = pd.DataFrame(counts, index=corpus["corpus"], columns=["none", "fewer", "all but one", "all"]).groupby(level=0).sum()
rest = (100 * totals["none"] / totals.sum(axis=1)).round(1)   # every voice silent: a pause of the whole texture, left out below
shares = totals[["all", "all but one", "fewer"]]
shares = 100 * shares.div(shares.sum(axis=1), axis=0)
stacked(shares, "share of time", "How many voices sing at once, as a share of the time when at least one sings: every voice, all but one, "
        f"or fewer. Every voice sings {shares.loc['bach', 'all']:.1f}% of the time in Bach and {shares.loc['palestrina', 'all']:.1f}% in Palestrina.",
        stem="03_texture")
shares.round(1).assign(files=corpus.groupby("corpus").size(), rest=rest).T

**Result**
- Every voice sings 99.8% of the time in Bach, 40.8% in Palestrina (all but one 35.2%, fewer 24.0%).
- So the model needs rests and voices that enter one after another, which the chorales almost never need.
- Every voice rests 2.9% of the time in Bach, 0.1% in Palestrina.